## LLM-as-a-Judge 프롬프트 설계 실험

설계 원칙 및 선행 연구 근거:

[Stage 1] 기본 Judge
  - Pointwise (Pass/Fail): UserSimCRS v2 (Afzali et al., WSDM 2023), G-Eval (Liu et al., EMNLP 2023)

[Stage 2] CoT 추가
  - CoT reasoning: Arize AI Blog (2025)
  - CoT + form-filling: G-Eval (Liu et al., EMNLP 2023)

[Stage 3] 루브릭 명시
  - Rule Augmentation: From Generation to Judgment (Li et al., EMNLP 2025)
  - 기준 분해: EvidentlyAI Blog (2025)
  - current_context 축 (Proactivity): ChatCRS (2024)
  - Preference Leakage 차단: Li et al., ICLR 2026

입력 (모든 Stage 공통): 페르소나 DNA + book_intros

In [1]:
# 라이브러리 설치
!pip install -q openai

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
# 라이브러리 임포트 및 환경 설정
import os
import json
from datetime import datetime, timezone, timedelta
from openai import OpenAI
from google.colab import userdata

os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

client    = OpenAI()
KST       = timezone(timedelta(hours=9))
LLM_MODEL = "gpt-4o-mini"

# PERSONA_BANK — Judge 평가 시 페르소나 DNA 참조용
PERSONA_BANK = {
    "A_최재원": {
        # CRS 슬롯 필드
        "reading_goal":     "이직을 준비하며 커리어 방향을 다시 잡는 데 도움이 될 통찰을 얻고 싶음",
        "preferred_genre":  "경제, 자기계발, 철학, 심리 (교양서)",
        "reading_style":    "핵심이 분명하고 실생활에 연결되는 글을 선호",
        "difficulty_level": "성인 일반 수준, 너무 두껍거나 학술적인 책은 부담",
        "current_context":  "평일 저녁 퇴근 후, 지치고 막막한 상태에서 방향을 찾고 싶음",
        # 보조 필드
        "demographics":     "32세, 스타트업 마케터(5년차), 서울 성동구 거주, 1인 가구",
        "disliked":         "다들 읽는 뻔한 베스트셀러, 전형적인 자기계발서",
        "pain_points":      "추천 근거 없으면 손이 안 감 / 선택지 많으면 못 고름",
        "speaking_style":   "구어체 존댓말, 망설임 표현 자연스럽게('그냥...', '음...'), 1~3문장",
    },
    "B_한미영": {
        "reading_goal":     "두 자녀(초5, 중1)에게 맞는 책을 찾아주고 싶음",
        "preferred_genre":  "자녀용: 초5는 역사·과학, 중1은 AI·로봇",
        "reading_style":    "아이가 흥미를 잃지 않고 끝까지 읽을 수 있는 구성",
        "difficulty_level": "각 자녀 학년 수준에 정확히 맞는 난이도",
        "current_context":  "자녀 교육 목적, 아이 수준 판단이 어려워 도움이 필요함",
        "demographics":     "43세, 전업주부(전직 초등교사), 경기도 수원시 거주",
        "disliked":         "아이 수준에 안 맞는 너무 어렵거나 유치한 책",
        "pain_points":      "수준 판단 기준 없음 / 두 아이 관심사가 달라 따로 추천 원함",
        "speaking_style":   "차분하고 정중한 구어체, 자녀 이야기를 구체적으로 설명",
    },
    "C_오민아": {
        "reading_goal":     "인디 게임 개발에 입문하고 싶음 (코딩 거의 처음)",
        "preferred_genre":  "게임 개발·코딩 (이전에는 웹소설·콘텐츠 크리에이터 관심)",
        "reading_style":    "단계적으로 따라갈 수 있는 입문서, 자기 언어로 시작",
        "difficulty_level": "고등학생 입문 수준, 너무 전문적이면 부담",
        "current_context":  "관심사가 빠르게 바뀌는 시기, 최신 트렌드 도서를 원함",
        "demographics":     "17세, 고등학생, 인천시 연수구 거주",
        "disliked":         "옛날 책, 지나치게 전문적인 기술서",
        "pain_points":      "검색해도 옛날 책만 나옴 / 자기 수준에 맞는지 판단 어려움",
        "speaking_style":   "10대 구어체, 솔직하고 직설적, 짧은 문장",
    },
}

### LLM Judge 프롬프트 정의

In [4]:
# Stage 1: 기본 Judge
# 최소한의 지시만 포함. 판단 기준을 Judge에게 맡김.
JUDGE_PROMPT_STAGE1 = """\
당신은 도서 추천 시스템의 독립적인 평가자입니다.

아래 사용자 프로파일(DNA)과 도서 소개글을 보고
각 도서가 이 사람에게 맞는 추천인지 판단하세요.

## 사용자 프로파일 (DNA)
{persona_str}

## 평가 대상 도서 소개글
{book_intros_str}

## 출력 형식 (JSON만 출력)
{{
  "books_evaluated": [
    {{
      "title":  "책 제목",
      "match":  true,
      "reason": "판단 근거 한 문장"
    }}
  ],
  "overall_reason": "전체 소감 한 문장"
}}
"""

In [5]:
# Stage 2: CoT 추가
# 판단 전 추론 과정을 먼저 서술하도록 강제 (자유 서술).
# 근거: Arize AI Blog (2025) — explanation 요청이 판단 일관성 향상
# 근거: G-Eval (Liu et al., EMNLP 2023) — CoT + form-filling 조합
JUDGE_PROMPT_STAGE2 = """\
당신은 도서 추천 시스템의 독립적인 평가자입니다.

아래 사용자 프로파일(DNA)과 도서 소개글을 보고
각 도서가 이 사람에게 맞는 추천인지 판단하세요.

## 사용자 프로파일 (DNA)
{persona_str}

## 평가 대상 도서 소개글
{book_intros_str}

## 판단 순서 (반드시 이 순서대로)
① 소개글을 읽고 이 책이 어떤 책인지 파악한다
② DNA와 대조해 맞는 이유, 맞지 않는 이유를 서술한다
③ 위 추론을 바탕으로 최종 판단을 내린다

## 출력 형식 (JSON만 출력)
{{
  "books_evaluated": [
    {{
      "title":     "책 제목",
      "reasoning": "① ~ ③ 사고 과정 자유 서술",
      "match":     true,
      "reason":    "최종 판단 한 문장"
    }}
  ],
  "overall_reason": "전체 소감 한 문장"
}}
"""

In [6]:
# Stage 3: 루브릭 명시
# 페르소나 DNA 5개 필드를 판단 축으로 구조화된 CoT 적용.
# 근거: From Generation to Judgment (Li et al., EMNLP 2025) — Rule Augmentation
# 근거: ChatCRS (2024) — current_context 축 (Proactivity 개념 차용)
# 근거: Preference Leakage (Li et al., ICLR 2026) — DNA + book_intro만 입력
JUDGE_PROMPT_STAGE3 = """\
당신은 도서 추천 시스템의 독립적인 평가자입니다.

아래 사용자 프로파일(DNA)과 도서 소개글을 보고
각 도서가 이 사람에게 맞는 추천인지 판단하세요.

## 사용자 프로파일 (DNA)
{persona_str}

## 평가 대상 도서 소개글
{book_intros_str}

## 판단 루브릭 (5개 축을 순서대로 확인)
1. reading_goal     : 소개글의 내용이 DNA의 독서 목적에 부합하는가?
2. preferred_genre  : 소개글의 주제/장르가 DNA의 선호 장르와 맞는가?
3. reading_style    : 소개글의 구성 방식이 DNA의 독서 스타일에 맞는가?
4. difficulty_level : 소개글의 난이도가 DNA의 수준에 적합한가?
5. current_context  : 지금 이 사람의 현재 상황에 실제로 도움이 되는가?

## 판단 순서 (반드시 이 순서대로)
① 루브릭 5개 축을 DNA와 대조해 각 한 문장씩 서술한다
② 5개 중 3개 이상 충족 시 match=true, 미만이면 match=false
③ 최종 판단 근거를 한 문장으로 요약한다

## 출력 형식 (JSON만 출력)
{{
  "books_evaluated": [
    {{
      "title": "책 제목",
      "reasoning": {{
        "reading_goal":     "목적 부합 여부 한 문장",
        "preferred_genre":  "장르 일치 여부 한 문장",
        "reading_style":    "독서 스타일 적합 여부 한 문장",
        "difficulty_level": "난이도 적합 여부 한 문장",
        "current_context":  "현재 상황 적합 여부 한 문장"
      }},
      "match":  true,
      "reason": "최종 판단 한 문장"
    }}
  ],
  "overall_reason": "전체 소감 한 문장"
}}
"""

In [37]:
# Stage 3: 루브릭 명시
# 페르소나 DNA 5개 필드를 판단 축으로 구조화된 CoT 적용.
# 근거: From Generation to Judgment (Li et al., EMNLP 2025) — Rule Augmentation
# 근거: ChatCRS (2024) — current_context 축 (Proactivity 개념 차용)
# 근거: Preference Leakage (Li et al., ICLR 2026) — DNA + book_intro만 입력
JUDGE_PROMPT_STAGE3_ts = """\
당신은 도서 추천 시스템의 독립적인 평가자입니다.

아래 사용자 프로파일(DNA)과 도서 소개글을 보고
각 도서가 이 사용자에게 적합한 추천인지 판단하세요.

## 사용자 프로파일 (DNA)
{persona_str}

## 평가 대상 도서 소개글
{book_intros_str}

## 판단 루브릭 (5개 축을 순서대로 확인)
1. reading_goal     : 소개글의 내용이 DNA의 독서 목적에 부합하는가?
2. preferred_genre  : 소개글의 주제/장르가 DNA의 선호 장르와 맞는가?
3. reading_style    : 소개글의 구성 방식이 DNA의 독서 스타일에 맞는가?
4. difficulty_level : 소개글의 난이도가 DNA의 수준에 적합한가?
5. current_context  : 지금 이 사람의 현재 상황에 실제로 도움이 되는가?

## 절대 규칙
- 반드시 소개글에 명시된 내용만을 근거로 판단한다
- 책 제목·저자로 알고 있는 외부 지식, 리뷰, 배경지식은 사용 금지다

## 판단 순서 (반드시 이 순서대로)
① 루브릭 5개 축을 각각 만족하는지 판단한다
② 5개 중 3개 이상 충족 시 match=true, 미만이면 match=false
③ 최종 판단 근거를 한 문장으로 요약한다

## 출력 형식 (JSON만 출력)
{{
  "books_evaluated": [
    {{
      "title": "책 제목",
      "match":  true,
      "reason": "판단 근거"

    }}
  ],
}}
"""

In [38]:
JUDGE_PROMPTS = {
    "stage1": JUDGE_PROMPT_STAGE1,
    "stage2": JUDGE_PROMPT_STAGE2,
    "stage3": JUDGE_PROMPT_STAGE3_ts,
}

### PeekaJudge 클래스

In [24]:
class PeekaJudge:
    """
    페르소나 DNA와 book_intro 기반으로 추천 도서 적합성을 평가하는 Judge.

    stage 파라미터로 Stage 1 ~ 3 중 선택.
    모든 Stage 공통 입력: 페르소나 DNA + book_intros
    """

    def __init__(self,
                 stage: str = "stage1",
                 model: str = LLM_MODEL,
                 verbose: bool = True):
        assert stage in JUDGE_PROMPTS, f"stage는 {list(JUDGE_PROMPTS.keys())} 중 하나여야 해."
        self.stage   = stage
        self.model   = model
        self.verbose = verbose

    def evaluate(self,
                 persona: dict,
                 book_intros: dict,
                 temperature: float = 0.0,
                 seed: int = 42) -> dict:
        """
        Args:
            persona    : 페르소나 DNA 딕셔너리
            book_intros: {"제목 | 저자": "소개글"} 형태
            temperature: 일관성 확보를 위해 0.0 기본값
            seed       : 재현성 (OpenAI best effort)

        Returns:
            {
              "books_evaluated": [...],
              "overall_reason":  "...",
              "book_match_rate":  0.0 ~ 1.0,
              "stage":           "stage1" | "stage2" | "stage3",
              "model":           "gpt-4o-mini",
              "evaluated_at":    "KST ISO 시간"
            }
        """
        persona_str     = "\n".join(f"- {k}: {v}" for k, v in persona.items())
        book_intros_str = "\n\n".join([
            f"📚 {title}\n소개: {intro}"
            for title, intro in book_intros.items()
        ])

        prompt = JUDGE_PROMPTS[self.stage].format(
            persona_str=persona_str,
            book_intros_str=book_intros_str
        )

        resp = client.chat.completions.create(
            model=self.model,
            messages=[
                {"role": "system", "content": prompt},
                {"role": "user",   "content": "위 도서 소개글을 평가해주세요."}
            ],
            temperature=temperature,
            seed=seed,
            response_format={"type": "json_object"},
        )

        content = resp.choices[0].message.content
        raw     = content.strip() if content else \
                  '{"books_evaluated": [], "overall_reason": "응답 없음"}'

        try:
            result = json.loads(raw)
        except json.JSONDecodeError:
            result = {"books_evaluated": [], "overall_reason": raw}

        books   = result.get("books_evaluated", [])
        matched = sum(1 for b in books if b.get("match"))

        result["book_match_rate"] = round(matched / len(books), 2) if books else 0.0
        result["stage"]           = self.stage
        result["model"]           = self.model
        result["evaluated_at"]    = datetime.now(tz=KST).isoformat()

        if self.verbose:
            print(f"\n[PeekaJudge — {self.stage} / {self.model}]")
            for b in books:
                mark = "O" if b.get("match") else "X"
                print(f"  [{mark}] {b.get('title', '?')} — {b.get('reason', '')}")
                # Stage 2: reasoning 자유 서술 출력
                if self.stage == "stage2" and b.get("reasoning"):
                    print(f"        reasoning: {b['reasoning']}")
                # Stage 3: reasoning 루브릭 축별 출력
                if self.stage == "stage3" and isinstance(b.get("reasoning"), dict):
                    for axis, desc in b["reasoning"].items():
                        print(f"        {axis}: {desc}")
            print(f"  총평: {result.get('overall_reason', '')}")
            print(f"  match_rate: {matched}/{len(books)}권 "
                  f"({result['book_match_rate']:.0%})")

        return result

### 책 소개글 추출 함수 정의

In [25]:
def extract_book_intros(session_result: dict) -> dict:
    """
    run_session() 결과에서 추천 3권의 book_intros 추출.
    run_stage_experiment()에 고정 입력으로 넘기기 위한 함수.
    """
    recommendations = session_result.get("recommendations", [])
    retrieved       = {
        b["isbn"]: b
        for b in session_result.get("retrieved_books", [])
        if b.get("isbn")
    }

    book_intros = {}
    for rec in recommendations:
        isbn   = rec.get("isbn", "")
        title  = rec.get("title", "")
        author = rec.get("author", "")
        key    = f"{title} | {author}"
        if isbn in retrieved and retrieved[isbn].get("book_intro"):
            book_intros[key] = retrieved[isbn]["book_intro"]

    return book_intros

### 실험 함수 정의

In [26]:
def run_stage_experiment(persona: dict,
                         book_intros: dict,
                         stages: list = None,
                         n_trials: int = 5) -> dict:
    """
    동일한 입력으로 Stage 1 ~ 3을 각각 n_trials회 실행해 일관성 비교.

    Args:
        persona    : 페르소나 DNA 딕셔너리
        book_intros: {"제목 | 저자": "소개글"} 형태
        stages     : 실험할 Stage 목록 (기본값: 전체)
        n_trials   : 반복 횟수 (기본값: 5)

    Returns:
        Stage별 일관성 요약 딕셔너리
    """
    if stages is None:
        stages = ["stage1", "stage2", "stage3"]

    results = {stage: [] for stage in stages}

    for stage in stages:
        print(f"\n{'='*50}")
        print(f"Stage: {stage} ({n_trials}회 반복)")
        print(f"{'='*50}")

        judge = PeekaJudge(stage=stage, verbose=False)

        for trial in range(1, n_trials + 1):
            result     = judge.evaluate(persona, book_intros)
            books      = result.get("books_evaluated", [])
            match_rate = result.get("book_match_rate", 0.0)

            book_summary = " | ".join([
                f"{'O' if b.get('match') else 'X'} {b.get('title', '?')}"
                for b in books
            ])
            print(f"  [{trial:02d}회] match_rate: {match_rate:.0%}  ->  {book_summary}")

            results[stage].append({
                "trial":      trial,
                "match_rate": match_rate,
                "books":      books
            })

    # 일관성 요약 출력
    print(f"\n{'='*50}")
    print("Stage별 일관성 요약")
    print(f"{'='*50}")

    summary = {}
    for stage in stages:
        rates = [r["match_rate"] for r in results[stage]]
        avg   = sum(rates) / len(rates)
        uniq  = len(set(rates))
        print(f"{stage:8s} | 평균: {avg:.0%} | "
              f"변동: {'일관' if uniq == 1 else f'{uniq}가지 다른 결과'}")
        summary[stage] = {
            "avg_match_rate":  round(avg, 2),
            "is_consistent":   uniq == 1,
            "unique_outcomes": uniq
        }

    return {"details": results, "summary": summary}

### Stage 간 비교 실험 함수 정의

In [27]:
def judge_session(session_result: dict,
                  persona: dict,
                  stage: str = "stage3",
                  verbose: bool = True) -> dict:
    """
    run_session() 결과에서 book_intros를 추출해 Judge 평가 실행.

    Args:
        session_result: run_session()의 return 딕셔너리
        persona       : PERSONA_BANK[persona_id]
        stage         : 사용할 Judge Stage (기본값: stage3)

    Returns:
        Judge 평가 결과 딕셔너리
    """
    recommendations = session_result.get("recommendations", [])
    retrieved       = {
        b["isbn"]: b
        for b in session_result.get("retrieved_books", [])
        if b.get("isbn")
    }

    book_intros = {}
    for rec in recommendations:
        isbn   = rec.get("isbn", "")
        title  = rec.get("title", "")
        author = rec.get("author", "")
        key    = f"{title} | {author}"
        if isbn in retrieved and retrieved[isbn].get("book_intro"):
            book_intros[key] = retrieved[isbn]["book_intro"]

    if not book_intros:
        print("  [Judge 스킵] book_intro 없음")
        return {}

    print(f"\n  [book_intros 로드 완료] {len(book_intros)}권")
    for title, intro in book_intros.items():
        print(f"    📚 {title}")
        print(f"       {intro[:60]}...")

    judge = PeekaJudge(stage=stage, verbose=verbose)
    return judge.evaluate(persona, book_intros)

### 실험 실행

In [13]:
# stage 1단계 샘플 테스트
with open("/content/drive/MyDrive/aiffel_final_pjt/notebooks/results/langfuse_v04_eval_mode_02.json",
          "r", encoding="utf-8") as f:
    data = json.load(f)

result  = data["results"]
persona = PERSONA_BANK[result["persona_id"]]

# Judge 실행 — book_intros 자동 추출
judge_result = judge_session(
    session_result=result,
    persona=persona,
    stage="stage1",
    verbose=True
)


  [book_intros 로드 완료] 3권
    📚 대한민국 30대, 재테크로 말하라 | 최성우
       30대는 이후의 인생을 위해 평생의 부 를 계획하고 관리해 나가야 할 시기다. 때문에 30대에게 있어 재테크...
    📚 30대 변화를 먹고 살아라 | 나카타니 아키히로
       오늘날 우리 사회의 30대는 어떤 모습인가? 이런저런 핑계를 대거나 뻔한 변명을 해대며 좀더 편안하고 쉬운 ...
    📚 30대에 성공하고 싶다면 자금관리에 미쳐라 | 박연수
       30대여 지금 무엇을 하든 성공과 행복을 꿈꾼다면 자금관리에 강한 사람이 되라! 30대의 성공, 문제는 결국...

[PeekaJudge — stage1 / gpt-4o-mini]
  [O] 대한민국 30대, 재테크로 말하라 — 재테크와 재무설계에 대한 실질적인 방법론을 제시하여 커리어 방향을 잡는 데 도움이 될 수 있습니다.
  [O] 30대 변화를 먹고 살아라 — 30대의 변화와 성공을 위한 질문을 던져주어 자기 성찰과 방향성을 찾는 데 유용할 것입니다.
  [O] 30대에 성공하고 싶다면 자금관리에 미쳐라 — 자금 관리와 성공의 관계를 명확히 설명하여 실생활에 적용할 수 있는 통찰을 제공합니다.
  총평: 모든 책이 커리어와 재정 관리에 대한 실질적인 통찰을 제공하여 이직 준비에 도움이 될 것 같습니다.
  match_rate: 3/3권 (100%)


In [ ]:
# 1회 반복 실행
json_files = [
    "/content/drive/MyDrive/aiffel_final_pjt/notebooks/results/langfuse_v04_eval_mode_01.json",
    "/content/drive/MyDrive/aiffel_final_pjt/notebooks/results/langfuse_v04_eval_mode_02.json",
    "/content/drive/MyDrive/aiffel_final_pjt/notebooks/results/langfuse_v04_eval_mode_03.json",
    "/content/drive/MyDrive/aiffel_final_pjt/notebooks/results/langfuse_v04_eval_mode_04.json",
    "/content/drive/MyDrive/aiffel_final_pjt/notebooks/results/langfuse_v04_eval_mode_05.json",
]

all_experiment_results = []

for filepath in json_files:
    filename = os.path.basename(filepath)

    with open(filepath, "r", encoding="utf-8") as f:
        data = json.load(f)

    result  = data["results"]
    persona = PERSONA_BANK[result["persona_id"]]

    print(f"\n{'='*50}")
    print(f"파일: {filename}")
    print(f"페르소나: {result['persona_id']}")
    print(f"{'='*50}")

    # Stage 1~3 순서대로 실행
    for stage in ["stage1", "stage2", "stage3"]:
        print(f"\n--- {stage} ---")
        judge_result = judge_session(
            session_result=result,
            persona=persona,
            stage=stage,
            verbose=True
        )

        if judge_result:
            all_experiment_results.append({
                "source_file":    filename,
                "persona_id":     result["persona_id"],
                "stage":          stage,
                "book_match_rate": judge_result.get("book_match_rate", 0.0),
                "books_evaluated": judge_result.get("books_evaluated", []),
                "overall_reason": judge_result.get("overall_reason", ""),
                "model":          judge_result.get("model", ""),
                "evaluated_at":   judge_result.get("evaluated_at", "")
            })

# 전체 요약 출력
print(f"\n{'='*50}")
print("실험 요약")
print(f"{'='*50}")
for r in all_experiment_results:
    print(f"  {r['source_file']} | {r['stage']} | "
          f"match_rate: {r['book_match_rate']:.0%}")

# JSON 저장
output = {
    "metadata": {
        "experiment":   "judge_prompt_stage_comparison",
        "stages":       ["stage1", "stage2", "stage3"],
        "json_files":   [os.path.basename(f) for f in json_files],
        "saved_at":     datetime.now(tz=KST).isoformat()
    },
    "results": all_experiment_results
}

save_path = (
    f"/content/drive/MyDrive/aiffel_final_pjt/notebooks/results/"
    f"judge_stage_experiment_"
    f"{datetime.now(tz=KST).strftime('%Y%m%d_%H%M%S')}.json"
)

with open(save_path, "w", encoding="utf-8") as f:
    json.dump(output, f, ensure_ascii=False, indent=2)

print(f"\n저장 완료: {save_path}")


파일: langfuse_v04_eval_mode_01.json
페르소나: A_최재원

--- stage1 ---

  [book_intros 로드 완료] 3권
    📚 대한민국 30대, 재테크로 말하라 | 최성우
       30대는 이후의 인생을 위해 평생의 부 를 계획하고 관리해 나가야 할 시기다. 때문에 30대에게 있어 재테크...
    📚 30대가 꼭 알아야 할 돈 관리법 32 | 한석희
       돈 관리법을 제대로 몰라 잘못된 재테크를 하고 있는 대한민국의 30대를 위한 책. 국제공인재무설계사(CFP)...
    📚 진짜 공부는 서른에 시작된다 - ‘생존’을 넘어 ‘성장’을 부르는 내 인생 공부 혁명 | 이창준
       토익에 자격증에 이러닝에, 업무와 자기계발 두 마리 토끼를 쫓는 데 몰두하느라 공부의 진정한 즐거움을 모르고...

[PeekaJudge — stage1 / gpt-4.1-2025-04-14]
  [X] 대한민국 30대, 재테크로 말하라 — 재테크 중심 내용이라 커리어 방향이나 자기 성찰에 대한 통찰은 부족해 보여요.
  [X] 30대가 꼭 알아야 할 돈 관리법 32 — 실질적인 돈 관리 팁 위주라 이직 고민이나 커리어 방향성에 직접적인 도움은 안 될 것 같아요.
  [O] 진짜 공부는 서른에 시작된다 - ‘생존’을 넘어 ‘성장’을 부르는 내 인생 공부 혁명 — 30대의 성장, 자기변화, 인생 방향성에 대한 실질적 조언이 많아서 이직 준비에 통찰을 줄 수 있을 것 같아요.
  총평: 세 권 중 두 권은 재테크 실용서라 커리어 고민에는 좀 동떨어져 있고, 마지막 책이 그나마 방향 찾는 데 실질적으로 도움 될 것 같네요.
  match_rate: 1/3권 (33%)

--- stage2 ---

  [book_intros 로드 완료] 3권
    📚 대한민국 30대, 재테크로 말하라 | 최성우
       30대는 이후의 인생을 위해 평생의 부 를 계획하고 관리해 나가야 할 시기다. 때문에 30대에게 있어 재테크...
   

In [21]:
# n회 반복 시행
json_files = [
    "/content/drive/MyDrive/aiffel_final_pjt/notebooks/results/langfuse_v04_eval_mode_01.json",
    "/content/drive/MyDrive/aiffel_final_pjt/notebooks/results/langfuse_v04_eval_mode_02.json",
    "/content/drive/MyDrive/aiffel_final_pjt/notebooks/results/langfuse_v04_eval_mode_03.json",
    "/content/drive/MyDrive/aiffel_final_pjt/notebooks/results/langfuse_v04_eval_mode_04.json",
    "/content/drive/MyDrive/aiffel_final_pjt/notebooks/results/langfuse_v04_eval_mode_05.json",
]

all_experiment_results = []

for filepath in json_files:
    filename = os.path.basename(filepath)

    with open(filepath, "r", encoding="utf-8") as f:
        data = json.load(f)

    result  = data["results"]
    persona = PERSONA_BANK[result["persona_id"]]

    # book_intros 한 번만 추출 — 모든 Stage에 동일한 입력으로 고정
    book_intros = extract_book_intros(result)

    if not book_intros:
        print(f"\n[스킵] {filename} — book_intro 없음")
        continue

    print(f"\n{'='*50}")
    print(f"파일: {filename}")
    print(f"페르소나: {result['persona_id']}")
    print(f"book_intros: {len(book_intros)}권")
    for title in book_intros:
        print(f"  📚 {title}")
    print(f"{'='*50}")

    # 동일한 book_intros로 Stage 1~3 비교 실험 (n_trials회 반복)
    experiment = run_stage_experiment(
        persona=persona,
        book_intros=book_intros,
        stages=["stage1", "stage2", "stage3"],
        n_trials=10
    )

    all_experiment_results.append({
        "source_file": filename,
        "persona_id":  result["persona_id"],
        "book_intros": list(book_intros.keys()),
        "experiment":  experiment
    })

# 전체 요약 출력
print(f"\n{'='*50}")
print("전체 실험 요약")
print(f"{'='*50}")
for r in all_experiment_results:
    print(f"\n[{r['source_file']}]")
    summary = r["experiment"].get("summary", {})
    for stage, s in summary.items():
        print(f"  {stage:8s} | 평균: {s['avg_match_rate']:.0%} | "
              f"변동: {'일관' if s['is_consistent'] else f'{s['unique_outcomes']}가지 다른 결과'}")

# JSON 저장
output = {
    "metadata": {
        "experiment": "judge_prompt_stage_comparison",
        "stages":     ["stage1", "stage2", "stage3"],
        "n_trials":   10,
        "json_files": [os.path.basename(f) for f in json_files],
        "saved_at":   datetime.now(tz=KST).isoformat()
    },
    "results": all_experiment_results
}

save_path = (
    f"/content/drive/MyDrive/aiffel_final_pjt/notebooks/results/"
    f"judge_stage_experiment_10_trials_"
    f"{datetime.now(tz=KST).strftime('%Y%m%d_%H%M%S')}.json"
)

with open(save_path, "w", encoding="utf-8") as f:
    json.dump(output, f, ensure_ascii=False, indent=2)

print(f"\n저장 완료: {save_path}")


파일: langfuse_v04_eval_mode_01.json
페르소나: A_최재원
book_intros: 3권
  📚 대한민국 30대, 재테크로 말하라 | 최성우
  📚 30대가 꼭 알아야 할 돈 관리법 32 | 한석희
  📚 진짜 공부는 서른에 시작된다 - ‘생존’을 넘어 ‘성장’을 부르는 내 인생 공부 혁명 | 이창준

Stage: stage1 (10회 반복)
  [01회] match_rate: 100%  ->  O 대한민국 30대, 재테크로 말하라 | O 30대가 꼭 알아야 할 돈 관리법 32 | O 진짜 공부는 서른에 시작된다 - ‘생존’을 넘어 ‘성장’을 부르는 내 인생 공부 혁명
  [02회] match_rate: 100%  ->  O 대한민국 30대, 재테크로 말하라 | O 30대가 꼭 알아야 할 돈 관리법 32 | O 진짜 공부는 서른에 시작된다 - ‘생존’을 넘어 ‘성장’을 부르는 내 인생 공부 혁명
  [03회] match_rate: 100%  ->  O 대한민국 30대, 재테크로 말하라 | O 30대가 꼭 알아야 할 돈 관리법 32 | O 진짜 공부는 서른에 시작된다 - ‘생존’을 넘어 ‘성장’을 부르는 내 인생 공부 혁명
  [04회] match_rate: 100%  ->  O 대한민국 30대, 재테크로 말하라 | O 30대가 꼭 알아야 할 돈 관리법 32 | O 진짜 공부는 서른에 시작된다 - ‘생존’을 넘어 ‘성장’을 부르는 내 인생 공부 혁명
  [05회] match_rate: 100%  ->  O 대한민국 30대, 재테크로 말하라 | O 30대가 꼭 알아야 할 돈 관리법 32 | O 진짜 공부는 서른에 시작된다 - ‘생존’을 넘어 ‘성장’을 부르는 내 인생 공부 혁명
  [06회] match_rate: 100%  ->  O 대한민국 30대, 재테크로 말하라 | O 30대가 꼭 알아야 할 돈 관리법 32 | O 진짜 공부는 서른에 시작된다 - ‘생존’을 넘어 ‘성장’을 부르는 내 인생 공부 혁명
  [07회] match_rate:

user_sim_test_v4_langfuse_eval_method_expt - 시뮬레이션 돌리고 대화 내역 + 추천 도서 3권 + 에이전트 셀프 만족도

In [40]:
# n회 반복 시행
json_files = [
    "/content/drive/MyDrive/aiffel_final_pjt/notebooks/results/langfuse_v04_eval_mode_01.json",
    "/content/drive/MyDrive/aiffel_final_pjt/notebooks/results/langfuse_v04_eval_mode_02.json",
    "/content/drive/MyDrive/aiffel_final_pjt/notebooks/results/langfuse_v04_eval_mode_03.json",
    "/content/drive/MyDrive/aiffel_final_pjt/notebooks/results/langfuse_v04_eval_mode_04.json",
    "/content/drive/MyDrive/aiffel_final_pjt/notebooks/results/langfuse_v04_eval_mode_05.json",
]

all_experiment_results = []

for filepath in json_files:
    filename = os.path.basename(filepath)

    with open(filepath, "r", encoding="utf-8") as f:
        data = json.load(f)

    result  = data["results"]
    persona = PERSONA_BANK[result["persona_id"]]

    # book_intros 한 번만 추출 — 모든 Stage에 동일한 입력으로 고정
    book_intros = extract_book_intros(result)

    if not book_intros:
        print(f"\n[스킵] {filename} — book_intro 없음")
        continue

    print(f"\n{'='*50}")
    print(f"파일: {filename}")
    print(f"페르소나: {result['persona_id']}")
    print(f"book_intros: {len(book_intros)}권")
    for title in book_intros:
        print(f"  📚 {title}")
    print(f"{'='*50}")

    # 동일한 book_intros로 Stage 1~3 비교 실험 (n_trials회 반복)
    experiment = run_stage_experiment(
        persona=persona,
        book_intros=book_intros,
        stages=["stage3"],
        n_trials=10
    )

    all_experiment_results.append({
        "source_file": filename,
        "persona_id":  result["persona_id"],
        "book_intros": list(book_intros.keys()),
        "experiment":  experiment
    })

# 전체 요약 출력
print(f"\n{'='*50}")
print("전체 실험 요약")
print(f"{'='*50}")
for r in all_experiment_results:
    print(f"\n[{r['source_file']}]")
    summary = r["experiment"].get("summary", {})
    for stage, s in summary.items():
        print(f"  {stage:8s} | 평균: {s['avg_match_rate']:.0%} | "
              f"변동: {'일관' if s['is_consistent'] else f'{s['unique_outcomes']}가지 다른 결과'}")

# JSON 저장
output = {
    "metadata": {
        "experiment": "judge_prompt_stage_comparison",
        "stages":     ["stage1", "stage2", "stage3"],
        "n_trials":   10,
        "json_files": [os.path.basename(f) for f in json_files],
        "saved_at":   datetime.now(tz=KST).isoformat()
    },
    "results": all_experiment_results
}

save_path = (
    f"/content/drive/MyDrive/aiffel_final_pjt/notebooks/results/"
    f"judge_stage_experiment_10_trials_"
    f"{datetime.now(tz=KST).strftime('%Y%m%d_%H%M%S')}.json"
)

with open(save_path, "w", encoding="utf-8") as f:
    json.dump(output, f, ensure_ascii=False, indent=2)

print(f"\n저장 완료: {save_path}")


파일: langfuse_v04_eval_mode_01.json
페르소나: A_최재원
book_intros: 3권
  📚 대한민국 30대, 재테크로 말하라 | 최성우
  📚 30대가 꼭 알아야 할 돈 관리법 32 | 한석희
  📚 진짜 공부는 서른에 시작된다 - ‘생존’을 넘어 ‘성장’을 부르는 내 인생 공부 혁명 | 이창준

Stage: stage3 (10회 반복)
  [01회] match_rate: 100%  ->  O 대한민국 30대, 재테크로 말하라 | O 30대가 꼭 알아야 할 돈 관리법 32 | O 진짜 공부는 서른에 시작된다 - ‘생존’을 넘어 ‘성장’을 부르는 내 인생 공부 혁명
  [02회] match_rate: 100%  ->  O 대한민국 30대, 재테크로 말하라 | O 30대가 꼭 알아야 할 돈 관리법 32 | O 진짜 공부는 서른에 시작된다 - ‘생존’을 넘어 ‘성장’을 부르는 내 인생 공부 혁명
  [03회] match_rate: 100%  ->  O 대한민국 30대, 재테크로 말하라 | O 30대가 꼭 알아야 할 돈 관리법 32 | O 진짜 공부는 서른에 시작된다 - ‘생존’을 넘어 ‘성장’을 부르는 내 인생 공부 혁명
  [04회] match_rate: 100%  ->  O 대한민국 30대, 재테크로 말하라 | O 30대가 꼭 알아야 할 돈 관리법 32 | O 진짜 공부는 서른에 시작된다 - ‘생존’을 넘어 ‘성장’을 부르는 내 인생 공부 혁명
  [05회] match_rate: 100%  ->  O 대한민국 30대, 재테크로 말하라 | O 30대가 꼭 알아야 할 돈 관리법 32 | O 진짜 공부는 서른에 시작된다 - ‘생존’을 넘어 ‘성장’을 부르는 내 인생 공부 혁명
  [06회] match_rate: 100%  ->  O 대한민국 30대, 재테크로 말하라 | O 30대가 꼭 알아야 할 돈 관리법 32 | O 진짜 공부는 서른에 시작된다 - ‘생존’을 넘어 ‘성장’을 부르는 내 인생 공부 혁명
  [07회] match_rate: